In [0]:
from pyspark.sql import functions as F


In [0]:
#the data and the variables
dbutils.widgets.text("run_date", "2026-06-10")
run_date = dbutils.widgets.get("run_date")
silver_day = (
    spark.table("coinwatch.silver.candles")
    .where(F.col("date") == run_date)
)


In [0]:
# create an aggregated daily gold metric
gold_daily = (
    silver_day
    .groupBy("symbol", "date")
    .agg(
        F.min_by("open","open_time").alias("day_open"),
        F.max_by("close","close_time").alias("day_close"),
        F.max("high").alias("day_high"),
        F.min("low").alias("day_low"),
        F.sum("volume").alias("day_volume"),
        F.sum("trade_count").alias("day_trades"),
        F.count("*").alias("minutes"),
    )
    .withColumn(
        "day_return_pct",
        (F.col("day_close")-F.col("day_open"))/F.col("day_open")*F.lit(100)
    )
)

In [0]:
#create gold table
spark.sql("CREATE SCHEMA IF NOT EXISTS coinwatch.gold")
#create the schema for gold table
(
    spark.createDataFrame([],gold_daily.schema)
    .write
    .format("delta")
    .mode("ignore")
    .saveAsTable("coinwatch.gold.daily_metrics")
)

In [0]:
#merge gold data in table
gold_daily.createOrReplaceTempView("gold_batch")

spark.sql("""
          merge into coinwatch.gold.daily_metrics as target
          using gold_batch as source
          on target.symbol = source.symbol
          and target.date = source.date
          when matched then update set *
          when not matched then insert *
          """)